# **PART 2 : TRAINING CONTEXTUAL QA MODELS**

## **Load Data**

In [ ]:
import pandas as pd
df = pd.read_csv(r"../Datasets/silver_standard_grp_3.csv")
df.columns

In [ ]:
df

## **PREPROCESSING**

### **Cleaning** 

#### **Remove unwanted Characters**

In [ ]:
import re

def clean_text(text):
    text = re.sub(r"\+", " ", text)
    text = re.sub(r"/+", " ", text)
    text = text.replace("_", "")
    text = text.replace("-", "")
    return text

In [ ]:
df[["context", "answer"]] = df[["context", "answer"]].apply(
    lambda col: col.map(clean_text)
)

#### **Remove null values**

In [ ]:
df.isna().sum()

#### **Drop duplicates**

In [ ]:
df.dropna(inplace=True,subset="question")

#### **Result**

In [ ]:
df

### **Tokenization**

In [ ]:
import re

def split_sentences(text):
    if not isinstance(text, str):
        return []

    sentences = re.split(r'[\.؟!\n،,]', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    return sentences

In [ ]:
df["list_content"]=df["context"].apply(split_sentences)

In [ ]:
df["list_content"]

In [ ]:
L =df["list_content"].apply(len)
lg = L.value_counts()
lg

In [ ]:
import matplotlib.pyplot as plt
plt.bar(lg.index, lg.values)
plt.xlabel("Longueur")
plt.ylabel("Fréquence")
plt.title("Distribution des phrases")
plt.show()

## **TRAINING**

### **Machine Learning classique**

#### **TF-IDF**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def predict_tfidf(context, question):
    sentences = split_sentences(context)

    corpus = sentences + [question]
    vectorizer = TfidfVectorizer().fit_transform(corpus)

    similarity = cosine_similarity(vectorizer[-1], vectorizer[:-1])

    best_idx = similarity.argmax()
    return sentences[best_idx]

In [ ]:
df["tfidf_pre"]=df.apply(lambda x : predict_tfidf(x["context"], x["question"]),axis=1)

#### **BM25**

In [ ]:
from rank_bm25 import BM25Okapi
def predict_bm25(context, question):
    sentences = split_sentences(context)

    tokenized_sentences = [s.split() for s in sentences]
    bm25 = BM25Okapi(tokenized_sentences)

    tokenized_question = question.split()
    scores = bm25.get_scores(tokenized_question)

    best_idx = scores.argmax()
    best_sentence = sentences[best_idx]

    return best_sentence

In [ ]:
df["BM25"]=df.apply(lambda x : predict_bm25(x["context"], x["question"]),axis=1)

#### **FastText**

In [ ]:
from gensim.models import FastText
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def sentence_vector(sentence, model):
    words = sentence.split()

    vectors = []
    for w in words:
        if w in model.wv:
            vectors.append(model.wv[w])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)
def predict_fasttext(context, question, model):
    sentences = split_sentences(context)

    q_vec = sentence_vector(question, model)

    scores = []

    for s in sentences:
        s_vec = sentence_vector(s, model)
        sim = cosine_similarity([q_vec], [s_vec])[0][0]
        scores.append(sim)

    best_idx = np.argmax(scores)
    return sentences[best_idx]

In [ ]:
sentences = [text.split() for text in df["context"]]

model = FastText(
    sentences,
    vector_size=200,
    window=7,
    min_count=1
)


In [ ]:
df["fasttext"]=df.apply(lambda x : predict_fasttext(x["context"], x["question"],model),axis=1)

#### **Results**

In [ ]:
df[["context", "question", "fasttext","BM25","tfidf_pre"]]

### **QA supervisée extractive (Transformers)**

##### **Split Data (80% train, 10% validation, 10% test)**

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)

val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

##### **Prepare Train, Validation, and Test data**

In [ ]:
#Transformer en format QA
def prepare_data(df):
    data = []
    for _, row in df.iterrows():
        context = row["context"]
        question = row["question"]
        answer = row["answer"]

        start = context.find(answer)

        if start == -1:
            continue

        data.append({
            "context": context,
            "question": question,
            "answers": {
                "text": [answer],
                "answer_start": [start]
            }
        })
    return data

train_data = prepare_data(train_df)
val_data   = prepare_data(val_df)
test_data  = prepare_data(test_df)

In [ ]:
#Convertir en Dataset HuggingFace
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)
test_dataset  = Dataset.from_list(test_data)

##### **Load Model**

In [ ]:
#Charger modèle + tokenizer
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_name = "UBC-NLP/marbertv2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

##### **Tokenization**

In [ ]:
#Tokenisation
def preprocess(example):
    inputs = tokenizer(
        example["question"],
        example["context"],
        truncation=True,
        padding="max_length",
        max_length=384,
        return_offsets_mapping=True
    )

    offsets = inputs["offset_mapping"]

    start_char = example["answers"]["answer_start"][0]
    end_char = start_char + len(example["answers"]["text"][0])

    start_token = 0
    end_token = 0

    for i, (start, end) in enumerate(offsets):
        if start <= start_char < end:
            start_token = i
        if start < end_char <= end:
            end_token = i

    inputs["start_positions"] = start_token
    inputs["end_positions"] = end_token

    inputs.pop("offset_mapping")

    return inputs

In [ ]:
train_dataset = train_dataset.map(preprocess)
val_dataset   = val_dataset.map(preprocess)
test_dataset  = test_dataset.map(preprocess)

##### **Train** 

In [ ]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
model.save_pretrained("/content/drive/MyDrive/qa_darija_model")
tokenizer.save_pretrained("/content/drive/MyDrive/qa_darija_model")

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/qa_darija_model")
model = AutoModelForQuestionAnswering.from_pretrained("/content/drive/MyDrive/qa_darija_model")

In [ ]:
import torch

def predict_answer(model, tokenizer, context, question):
    inputs = tokenizer(
        question,
        context,
        truncation=True,
        padding="max_length",
        max_length=384,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits) + 1

    input_ids = inputs["input_ids"][0]
    answer = tokenizer.decode(input_ids[start_idx:end_idx], skip_special_tokens=True)

    return answer

In [ ]:
test_df = test_df.reset_index(drop=True)

test_df["pred_answer"] = test_df.apply(
    lambda x: predict_answer(model, tokenizer, x["context"], x["question"]),
    axis=1
)

In [ ]:
#Exact Match
import re

def normalize(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def exact_match(pred, gold):
    return int(normalize(pred) == normalize(gold))

test_df["EM"] = test_df.apply(
    lambda x: exact_match(x["pred_answer"], x["answer"]),
    axis=1
)

em_score = test_df["EM"].mean()
print("Exact Match:", em_score)

In [ ]:
'Nous utilisons une métrique F1 basée sur le chevauchement de tokens, conforme au standard SQuAD, plutôt que le F1 de scikit-learn qui est destiné à la classification.'
from collections import Counter

def f1_score(pred, gold):
    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

test_df["F1"] = test_df.apply(
    lambda x: f1_score(x["pred_answer"], x["answer"]),
    axis=1
)

print("F1-score:", test_df["F1"].mean())

In [ ]:
test_df["boundary_error"] = test_df.apply(
    lambda x: normalize(x["answer"]) in normalize(x["pred_answer"]) and x["EM"] == 0,
    axis=1
)

In [ ]:
def detect_type_error(row):
    pred = row["pred_answer"]
    gold = row["answer"]

    if gold.isdigit() and not pred.isdigit():
        return True
    return False

test_df["type_error"] = test_df.apply(detect_type_error, axis=1)

In [ ]:
def detect_language(text):
    arabic = any('\u0600' <= c <= '\u06FF' for c in text)
    latin = any(c.isascii() and c.isalpha() for c in text)

    if arabic and latin:
        return "mixed"
    elif arabic:
        return "arabic"
    else:
        return "latin"

test_df["lang"] = test_df["question"].apply(detect_language)

test_df.groupby("lang")[["EM", "F1"]].mean()

In [ ]:
errors = test_df[test_df["EM"] == 0][
    ["question", "context", "answer", "pred_answer"]
].head(10)

errors

In [ ]:
results = {
    "TF-IDF": test_df.apply(lambda x: exact_match(x["tfidf_pre"], x["answer"]), axis=1).mean(),
    "BM25": test_df.apply(lambda x: exact_match(x["BM25"], x["answer"]), axis=1).mean(),
    "FastText": test_df.apply(lambda x: exact_match(x["fasttext"], x["answer"]), axis=1).mean(),
    "Transformer": test_df["EM"].mean()
}

results

In [ ]:
import matplotlib.pyplot as plt

plt.bar(results.keys(), results.values())
plt.title("Comparaison des modèles QA")
plt.ylabel("Exact Match")
plt.show()